## ETL Process on NYC Restaurants Data

In [1]:
import requests
import numpy as np
import pandas as pd
import sqlite3

pd.set_option('display.max_columns', None)

#Connecting to SQL
connector = sqlite3.connect("NY_Restaurants.db")
cursor = connector.cursor()

In [ ]:
#Making the tables in SQLite

cursor.execute("""
CREATE TABLE restaurants (
camis INTEGER,
dba TEXT,
boro TEXT CHECK (boro IN ('Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island')),
building TEXT,
street TEXT,
zipcode INTEGER,
phone INTEGER,
cuisine_description TEXT,
latitude REAL,
longitude REAL,
community_board INTEGER,
council_district INTEGER,
census_tract INTEGER,
bin INTEGER,
bbl INTEGER,
nta TEXT,
PRIMARY KEY (camis)
)
""")
connector.commit()

cursor.execute("""
    CREATE TABLE inspections (
    inspection_id INTEGER,
    restaurant_id INTEGER,
    grade TEXT CHECK (grade IN ('N', 'A', 'B', 'C', 'Z', 'P')),
    grade_date INTEGER,
    inspection_date INTEGER,
    action TEXT,
    violation_code TEXT,
    violation_description TEXT,
    critical_flag TEXT CHECK (critical_flag IN ('Critical', 'Not Critical', 'Not Applicable')),
    score INTEGER,
    inspection_type TEXT,
    PRIMARY KEY (inspection_id),
    FOREIGN KEY (restaurant_id) REFERENCES restaurants(camis)
    )
""")
connector.commit()
cursor.execute("""
CREATE TABLE cuisine_types (
id INTEGER,
restaurant_id INTEGER NOT NULL,
cuisine_type TEXT,
PRIMARY KEY (id),
FOREIGN KEY (restaurant_id) REFERENCES restaurants(camis)
)
""")
connector.commit()

### Data Extraction wiht API

In [29]:
#Downloading new data

data = []
limit = 50000
offset = 0

while True:
    response = requests.get("https://data.cityofnewyork.us/resource/43nn-pn8j.json", params = {"$limit": limit, "$offset": offset, "$order":":id"})
    round = response.json()

    if not round: break
    data.extend(round)

    offset += limit

data = pd.DataFrame(data)


### Data Cleaning

In [31]:
data.drop(columns = ["location", "record_date"] , inplace = True) # redundand column

data["phone"] = data["phone"].str.replace (" ", "", regex = False)
data["phone"] = data["phone"].where((~data["phone"].str.startswith(("0", "1"), na=False)) &
                                    (~data["phone"].str.contains("_", na=False)) &
                                    (data["phone"].str.len() == 10), np.nan)

#Date handling
data[["inspection_date", "grade_date"]] = data[["inspection_date", "grade_date"]].apply(pd.to_datetime, errors = "coerce")
data[["inspection_date", "grade_date"]] = data[["inspection_date", "grade_date"]].map(lambda x: x.to_julian_date() if pd.notnull(x) else None)
data[["inspection_date", "grade_date"]] = np.floor(data[["inspection_date", "grade_date"]]).astype("Int64")

data["inspection_date"] = data["inspection_date"].where(data["inspection_date"] != 2415020, np.nan)
data[["dba", "building", "street"]] = data[["dba", "building", "street"]].replace("N/A", None) #False NA-s
data["cuisine_description"] = data["cuisine_description"].replace('Not Listed/Not Applicable', 'Other')
data["boro"] = data["boro"].replace("0", None)

data["inspection_type"] = data["inspection_type"].str.title().str.strip().str.replace(r"\s*/\s*", " / ", regex = True)

data[["latitude", "longitude"]] = data[["latitude", "longitude"]].apply(pd.to_numeric)
data[["camis", "zipcode", "score", "community_board", "council_district", "census_tract", "bin", "bbl"]] = data[["camis", "zipcode", "score", "community_board", "council_district", "census_tract", "bin", "bbl"]].apply(pd.to_numeric).astype("Int64")

### Data Loading

In [33]:
#Making a dataframe for each table
restaurants_variables = ["dba", "boro", "building", "street", "zipcode", "phone", "cuisine_description", "latitude",
                         "longitude", "community_board", "council_district", "census_tract", "bin", "bbl", "nta"]
restaurants = data[["camis"] + restaurants_variables].copy()
restaurants.drop_duplicates(subset = "camis", keep = "last", inplace = True)

inspections = data.drop(columns = restaurants_variables)
inspections.rename(columns = {"camis": "restaurant_id"}, inplace = True)


restaurants.to_sql("restaurants", connector, if_exists = "append", index = False)
inspections.to_sql("inspections", connector, if_exists = "append", index = False)


296738

In [34]:
# Cuisine types correction

cuisine_types = restaurants[["camis", "cuisine_description"]].copy()
cuisine_types["cuisine_description"] = cuisine_types["cuisine_description"].str.split("/")
cuisine_types = cuisine_types.explode("cuisine_description")
cuisine_types["cuisine_description"] = cuisine_types["cuisine_description"].str.strip()
cuisine_types.rename(columns = {"camis": "restaurant_id", "cuisine_description": "cuisine_type"}, inplace = True)

cuisine_types.to_sql("cuisine_types", connector, if_exists = "append", index = False)

35387

### Table Reset

In [27]:
cursor.execute("DELETE FROM restaurants")
cursor.execute("DELETE FROM inspections")
cursor.execute("DELETE FROM cuisine_types")
connector.commit()

In [ ]:
connector.close()

### Notes

In [25]:
print(data.shape, end = "\n\n") #less rows than website mentions
print(data.isnull().sum()) # large amount of Null values in some variables


(296738, 25)

camis                         0
dba                           3
boro                         80
building                    976
street                        1
zipcode                    3127
phone                       397
cuisine_description        3267
inspection_date            3259
action                     3259
violation_code             5552
violation_description      5553
critical_flag                 0
score                     16363
inspection_type            3259
latitude                   1356
longitude                  1356
community_board            4473
council_district           4446
census_tract               4446
bin                        5765
bbl                        1356
nta                        4473
grade                    150369
grade_date               159718
dtype: int64


In [21]:
print(f"Observed restaurants number: {len(data["camis"].unique())}")

Observed restaurants number: 30699


In [19]:
data["cuisine_description"].sort_values().unique() # more labels to a restaurant --> SET in MySQL

array(['Afghan', 'African', 'American', 'Armenian', 'Asian/Asian Fusion',
       'Australian', 'Bagels/Pretzels', 'Bakery Products/Desserts',
       'Bangladeshi', 'Barbecue', 'Basque', 'Bottled Beverages',
       'Brazilian', 'Cajun', 'Californian', 'Caribbean', 'Chicken',
       'Chilean', 'Chimichurri', 'Chinese', 'Chinese/Cuban',
       'Chinese/Japanese', 'Coffee/Tea', 'Continental', 'Creole',
       'Creole/Cajun', 'Czech', 'Donuts', 'Eastern European', 'Egyptian',
       'English', 'Ethiopian', 'Filipino', 'French', 'Frozen Desserts',
       'Fruits/Vegetables', 'Fusion', 'German', 'Greek', 'Hamburgers',
       'Haute Cuisine', 'Hawaiian', 'Hotdogs', 'Hotdogs/Pretzels',
       'Indian', 'Indonesian', 'Iranian', 'Irish', 'Italian', 'Japanese',
       'Jewish/Kosher', 'Juice, Smoothies, Fruit Salads', 'Korean',
       'Latin American', 'Lebanese', 'Mediterranean', 'Mexican',
       'Middle Eastern', 'Moroccan', 'New American', 'New French',
       'Nuts/Confectionary', 'Other', 'P

In [41]:
data.head()

,camis,dba,boro,building,street,zipcode,phone,cuisine_description,inspection_date,action,violation_code,violation_description,critical_flag,score,inspection_type,latitude,longitude,community_board,council_district,census_tract,bin,bbl,nta,grade,grade_date
0,50098704,MEMORIAL SLOAN KETTERING - PLAZA CAFE,Manhattan,530,EAST 74 STREET,10021,2032477418,Coffee/Tea,2460759,Violations were cited in the following area(s).,04H,"Raw, cooked or prepared food is adulterated, c...",Critical,32,Cycle Inspection / Initial Inspection,40.767803,-73.951956,108,5,12400,1090713,1014850015,MN31,NaN,<NA>
1,50057884,PAN-TASTIC,Queens,42-02,NORTHERN BOULEVARD,11101,7187865280,Korean,2460275,Violations were cited in the following area(s).,06C,"Food, supplies, or equipment not protected fro...",Critical,8,Cycle Inspection / Initial Inspection,40.753068,-73.921378,401,26,17100,4002366,4001830051,QN31,A,2460275
2,50088036,YUM YUM TOO,Manhattan,662,9 AVENUE,10036,2039939934,Thai,2460100,Violations were cited in the following area(s).,08A,Establishment is not free of harborage or cond...,Not Critical,56,Cycle Inspection / Initial Inspection,40.761074,-73.990683,104,3,12700,1025038,1010370001,MN15,NaN,<NA>
3,50150567,FLAVORS & CATERING EVENTS,Manhattan,321,WEST 37 STREET,10018,9178216161,American,2461025,Violations were cited in the following area(s).,06C,"Food, supplies, or equipment not protected fro...",Critical,47,Cycle Inspection / Initial Inspection,40.754445,-73.992904,104,3,11100,1013618,1007610022,MN13,NaN,<NA>
4,50075315,TRIPLE J PIZZERIA,Bronx,1739,EAST 172 STREET,10472,7188603446,Pizza,2459998,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,27,Cycle Inspection / Initial Inspection,40.832539,-73.869564,209,18,7600,2027726,2038730001,BX08,NaN,<NA>
